<a href="https://colab.research.google.com/github/NataliiaFakas/TFG_BMW/blob/main/8_BN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Naive


---
## 1. Cargar los datos


In [1]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

drive.mount('/content/drive')
ruta_macro = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_variables_macro.csv"
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/ventas_bmw.csv"

df_macro = pd.read_csv(ruta_macro)
df_bmw = pd.read_csv(ruta_bmw)

print("Dataset macroeconómico:")
display(df_macro.head())

print("\nDataset de ventas de BMW:")
display(df_bmw.head())

Mounted at /content/drive
Dataset macroeconómico:


,Year,Unemployment_Rate,GDP_Growth,Deposit_Facility,HICP,Industrial_Production,Retail_Sales_Growth,Consumer_Confidence,Compensation_Per_Employee,Brent_Oil_Price,EUR_USD,Euribor_12M,Population_Total
0,2005,8.9,1.7,1.25,2.18,153.2,0.03,-4.2,31.22,54.57,1.2441,2.19,434585887
1,2006,8.2,3.2,2.50,2.19,147.7,0.04,-3.5,31.95,65.16,1.2556,3.08,436041018
2,2007,7.2,2.9,3.00,2.13,145.5,0.03,-2.1,32.75,72.44,1.3705,4.24,437514477
3,2008,7.1,0.4,2.00,3.30,148.0,0.01,-10.8,33.86,96.94,1.4708,4.63,439074280
4,2009,9.0,-4.4,0.25,0.29,144.5,-0.03,-21.7,34.41,61.74,1.3948,1.31,440467898



Dataset de ventas de BMW:


,Año,Ventas_BMW_Unidades
0,2005,1126768
1,2006,1185088
2,2007,1276793
3,2008,1202239
4,2009,1068770


## 2. Seleccionar y unir las variables del modelo

## 3. Unir las variables macroeconómicas con las ventas de BMW

In [2]:
def añadir_variables_macro(df_bmw, df_macro):
    df_final = df_bmw.merge(
        df_macro[variables_modelo],
        on="Year",
        how="left"
    )
    return df_final


variables_modelo = [
    "Year",
    "Compensation_Per_Employee",
    "Industrial_Production",
    "Population_Total",
    "EUR_USD"
]

df_bmw_filtrado = df_bmw.rename(columns={"Año": "Year"})

data = añadir_variables_macro(
    df_bmw_filtrado,
    df_macro
)

print("Dataset BMW:")
display(df_bmw_filtrado.head())

print("Dimensiones del dataset final:", data.shape)

display(data.head())

Dataset BMW:


,Year,Ventas_BMW_Unidades
0,2005,1126768
1,2006,1185088
2,2007,1276793
3,2008,1202239
4,2009,1068770


Dimensiones del dataset final: (21, 6)


,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,1126768,31.22,153.2,434585887,1.2441
1,2006,1185088,31.95,147.7,436041018,1.2556
2,2007,1276793,32.75,145.5,437514477,1.3705
3,2008,1202239,33.86,148.0,439074280,1.4708
4,2009,1068770,34.41,144.5,440467898,1.3948


## 4. Comprobar valores nulos

In [3]:
print("Valores nulos por variable:")

display(
    data.isnull().sum()
)

Valores nulos por variable:


,0
Year,0
Ventas_BMW_Unidades,0
Compensation_Per_Employee,0
Industrial_Production,0
Population_Total,0
EUR_USD,0


## 5. Partición temporal

Utilizamos exactamente la misma división que en los demás modelos:

- Train: 2005–2018
- Dev: 2019–2021
- Test: 2022–2025

En el caso del Baseline Naive, no existe una fase de entrenamiento, ya que no se estiman parámetros. Sin embargo, mantenemos la misma división temporal para que la evaluación sea homogénea con el resto de modelos.

In [4]:
data = data.sort_values("Year").reset_index(drop=True)

display(data)

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,1126768,31.22,153.2,434585887,1.2441
1,2006,1185088,31.95,147.7,436041018,1.2556
2,2007,1276793,32.75,145.5,437514477,1.3705
3,2008,1202239,33.86,148.0,439074280,1.4708
4,2009,1068770,34.41,144.5,440467898,1.3948
5,2010,1224280,35.20,127.2,441160594,1.3257
6,2011,1380384,35.97,128.9,440526112,1.3920
7,2012,1540085,36.55,125.5,441280191,1.2848
8,2013,1655138,37.13,120.7,441739935,1.3281
9,2014,1811719,37.64,116.5,442235565,1.3285


In [5]:
data_baseline = data.copy()

data_baseline["Prediccion_Naive"] = (
    data_baseline["Ventas_BMW_Unidades"].shift(1)
)

display(
    data_baseline[
        [
            "Year",
            "Ventas_BMW_Unidades",
            "Prediccion_Naive"
        ]
    ]
)

,Year,Ventas_BMW_Unidades,Prediccion_Naive
0,2005,1126768,NaN
1,2006,1185088,1126768.0
2,2007,1276793,1185088.0
3,2008,1202239,1276793.0
4,2009,1068770,1202239.0
5,2010,1224280,1068770.0
6,2011,1380384,1224280.0
7,2012,1540085,1380384.0
8,2013,1655138,1540085.0
9,2014,1811719,1655138.0


In [6]:
train_baseline = data_baseline[
    data_baseline["Year"] <= 2018
].copy()

dev_baseline = data_baseline[
    (data_baseline["Year"] >= 2019) &
    (data_baseline["Year"] <= 2021)
].copy()

test_baseline = data_baseline[
    data_baseline["Year"] >= 2022
].copy()

print("Observaciones Train:", len(train_baseline))
print("Observaciones Dev:", len(dev_baseline))
print("Observaciones Test:", len(test_baseline))

Observaciones Train: 14
Observaciones Dev: 3
Observaciones Test: 4


In [8]:
print(
    f"Train: {train_baseline['Year'].min()} - "
    f"{train_baseline['Year'].max()}"
)

print(
    f"Dev: {dev_baseline['Year'].min()} - "
    f"{dev_baseline['Year'].max()}"
)

print(
    f"Test: {test_baseline['Year'].min()} - "
    f"{test_baseline['Year'].max()}"
)

Train: 2005 - 2018
Dev: 2019 - 2021
Test: 2022 - 2025


## 6. Predicciones sobre dev

In [9]:
y_dev_naive = dev_baseline[
    "Ventas_BMW_Unidades"
]

pred_dev_naive = dev_baseline[
    "Prediccion_Naive"
]

mask_dev = (
    y_dev_naive.notna() &
    pred_dev_naive.notna()
)

y_dev_naive = y_dev_naive[mask_dev]
pred_dev_naive = pred_dev_naive[mask_dev]

In [10]:
mae_naive_dev = mean_absolute_error(
    y_dev_naive,
    pred_dev_naive
)

mse_naive_dev = mean_squared_error(
    y_dev_naive,
    pred_dev_naive
)

rmse_naive_dev = np.sqrt(
    mse_naive_dev
)

r2_naive_dev = r2_score(
    y_dev_naive,
    pred_dev_naive
)

print("RESULTADOS BASELINE NAIVE - DEV")
print("--------------------------------")
print(f"MAE:  {mae_naive_dev:,.2f}")
print(f"MSE:  {mse_naive_dev:,.2f}")
print(f"RMSE: {rmse_naive_dev:,.2f}")
print(f"R²:   {r2_naive_dev:.4f}")

RESULTADOS BASELINE NAIVE - DEV
--------------------------------
MAE:  122,827.67
MSE:  18,575,566,348.33
RMSE: 136,292.21
R²:   -1.9915


## 7. Predicciones sobre test

In [11]:
y_naive_test = test_baseline[
    "Ventas_BMW_Unidades"
]

pred_naive_test = test_baseline[
    "Prediccion_Naive"
]

mask_test = (
    y_naive_test.notna() &
    pred_naive_test.notna()
)

y_naive_test = y_naive_test[mask_test]
pred_naive_test = pred_naive_test[mask_test]

In [12]:
mae_naive_test = mean_absolute_error(
    y_naive_test,
    pred_naive_test
)

mse_naive_test = mean_squared_error(
    y_naive_test,
    pred_naive_test
)

rmse_naive_test = np.sqrt(
    mse_naive_test
)

r2_naive_test = r2_score(
    y_naive_test,
    pred_naive_test
)

In [13]:
print("RESULTADOS BASELINE NAIVE - TEST")
print("---------------------------------")
print(f"MAE:  {mae_naive_test:,.2f}")
print(f"MSE:  {mse_naive_test:,.2f}")
print(f"RMSE: {rmse_naive_test:,.2f}")
print(f"R²:   {r2_naive_test:.4f}")

RESULTADOS BASELINE NAIVE - TEST
---------------------------------
MAE:  87,580.00
MSE:  10,012,345,269.50
RMSE: 100,061.71
R²:   -2.2698


## 8. Tabla de predicciones del baseline

In [14]:
predicciones_naive = test_baseline[
    [
        "Year",
        "Ventas_BMW_Unidades",
        "Prediccion_Naive"
    ]
].copy()

predicciones_naive["Error"] = (
    predicciones_naive["Ventas_BMW_Unidades"]
    - predicciones_naive["Prediccion_Naive"]
)

predicciones_naive["Error_Absoluto"] = (
    predicciones_naive["Error"].abs()
)

predicciones_naive["Error_Porcentual"] = (
    predicciones_naive["Error_Absoluto"]
    / predicciones_naive["Ventas_BMW_Unidades"]
    * 100
)

display(predicciones_naive)

,Year,Ventas_BMW_Unidades,Prediccion_Naive,Error,Error_Absoluto,Error_Porcentual
17,2022,2100692,2213795.0,-113103.0,113103.0,5.384083
18,2023,2253835,2100692.0,153143.0,153143.0,6.794774
19,2024,2200177,2253835.0,-53658.0,53658.0,2.438804
20,2025,2169761,2200177.0,-30416.0,30416.0,1.401813


## 9. Obtener las cuatro métricas juntas

In [15]:
resultado_naive = pd.DataFrame({
    "Modelo": ["Baseline Naive"],
    "MAE": [mae_naive_test],
    "RMSE": [rmse_naive_test],
    "R²": [r2_naive_test]
})

display(resultado_naive)

,Modelo,MAE,RMSE,R²
0,Baseline Naive,87580.0,100061.707309,-2.269783
